In [1]:
### add logic not to sync reject-phase each time (should make the load a little lighter)

In [2]:
import pandas as pd
import json
import os
from src.config.config_prompt_tree import get_component_text, tech_dict, phase_dict
from src.config.config_paths import SUMMARIES_EVOLUTIONARY_CONTRASTIVE
from functions.utils import load_mappings, load_sources_dict, get_source_by_id
from functions.general import save_json_to_file, load_json, load_jsonl 
from functions.obsidian import to_json_list, return_validated_df, return_model_df, clean_model_df, get_value_by_id
from src.config.config_paths import PROJECTS_CHRONOLOGICAL, ARTICLE_ID_DATE_JSONL, ARTICLE_ID_TITLE_JSONL, ARTICLE_ID_URL_JSONL, OBSIDIAN_VAULT_PHASE

# CONFIGURATION
os.makedirs(OBSIDIAN_VAULT_PHASE, exist_ok=True)

#read dictionaries for mapping company, location and tech
companyID_to_company, locationID_to_location = load_mappings()
tech_id_to_tech = {v: k for k, v in tech_dict.items()}

#read in project-articles mapping for appending articles to markdown for each projectID
with open(PROJECTS_CHRONOLOGICAL, 'r') as file:
    project_articles = json.load(file)

article_date_dict = load_jsonl(ARTICLE_ID_DATE_JSONL)
article_title_dict = load_jsonl(ARTICLE_ID_TITLE_JSONL)
source_dict = load_sources_dict('src/source_ID.json')
article_url_dict = load_jsonl(ARTICLE_ID_URL_JSONL)

#return validated_df if it exists
validated_df, num_files = return_validated_df(OBSIDIAN_VAULT_PHASE)
print(f"Found {num_files} markdown files")

#return latest model_df
model_df = return_model_df('src/outputs/model/phase_level')

#clean model_df (map dictionaries; add NA/False checked | reject | etc columns)
model_df = clean_model_df(model_df, companyID_to_company, locationID_to_location)

# only pass projects that are in project_articles
model_df = model_df[model_df.index.astype(str).str[:15].isin(project_articles.keys())]

#set as output_df and update with validated entries for all cases
output_df = model_df.copy()
output_df.update(validated_df)
output_df.replace("Not disclosed", "-", inplace=True)

#return validated dictionaries for projectID, reject_phase, finetune, duplicate-project
if not validated_df.empty:
    #return projectID
    query_validated_df = validated_df.copy()
    query_validated_df['projectID'] = query_validated_df.index.astype(str).str[:15]

    #return phaseID_reject dictionary
    reject_phase_df = query_validated_df[query_validated_df['reject-phase'] == True].copy()
    reject_phase_dict = dict(zip(reject_phase_df['projectID'], reject_phase_df['reject-phase']))
    save_json_to_file(reject_phase_dict, 'src/inputs/validated_reject_phase.json')

    #return projectID_finetune dictionary
    finetune_df = query_validated_df[query_validated_df['finetune'] == True].copy()
    finetune_dict = dict(zip(finetune_df['projectID'], finetune_df['finetune']))
    save_json_to_file(finetune_dict, 'src/inputs/validated_finetune.json')
else:
    print("INITIAL LOAD: validated_df is empty")

#update vault content with output_df
for phaseID, row in output_df.iterrows():

    #get projectID from phase_id
    projectID = phaseID[:15]

    #load the project summary
    master_summary_path = f'{SUMMARIES_EVOLUTIONARY_CONTRASTIVE}/{projectID}.txt'
    with open(master_summary_path, 'r') as file:
        project_summary = file.read()

    #construct output filename and filepath
    filename = f"{phaseID}.md"
    filepath = os.path.join(OBSIDIAN_VAULT_PHASE, filename)
    
    #construct YAML frontmatter
    yaml_frontmatter = "---\n"
    #every column is mapped directly to the frontmatter with its value for row == projectID
    for column_name in output_df.columns:
        value = row[column_name]
        # Convert value to string in case it's numeric or another data type
        yaml_frontmatter += f"{column_name}: {value}\n"
    yaml_frontmatter += "---\n\n"

    #add summary text
    yaml_frontmatter += f"## {projectID}\n\n"
    yaml_frontmatter += f"{project_summary}\n\n"

    #add article references
    yaml_frontmatter += f"## Article References\n\n"
    articles = project_articles[projectID]
    for article in reversed(articles):
        article = str(article)
        date = get_value_by_id(article_date_dict, "date", article)
        title = get_value_by_id(article_title_dict, "title", article)
        source = get_source_by_id(source_dict, int(article[:2]))
        url = get_value_by_id(article_url_dict, "url", article)
        yaml_frontmatter += f"- {date}, {title}, [{source}]({url})\n"

    #write markdown file to obsidian vault
    with open(filepath, 'w', encoding='utf-8') as f:
        f.write(yaml_frontmatter)

/Users/ben/Documents/bruegel/data_new/WORKING/INDUSTRY/tuone_v6/functions/obsidian.py:90: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df.replace({'true': True, 'false': False, "True": True, "False": False}, inplace=True)


[False  True]
Validated dataframe detected with  277 projects:
                         country                    location        company  \
filename                                                                      
ESP-01932-01118_01_03_00     ESP                     Almaraz      Iberdrola   
BEL-06487-08471_01_00_00     BEL                     Seraing      CMI Group   
ITA-06900-07509_01_00_00     ITA  Northern and Central Italy          Terna   
ITA-00172-00848_01_01_00     ITA                      Cesano  Vulcan Energy   
BEL-06308-01626_01_00_00     BEL                   Vilvoorde          Engie   

                             tech           component  
filename                                               
ESP-01932-01118_01_03_00  battery          deployment  
BEL-06487-08471_01_00_00  battery          deployment  
ITA-06900-07509_01_00_00  battery          deployment  
ITA-00172-00848_01_01_00  battery  materials_refining  
BEL-06308-01626_01_00_00  battery          depl